# Bank Distress Early-Warning Model — Modeling
**MSDS 696 Data Science Practicum II · Oussama Ennaciri**

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

PROCESSED = Path("..") / "data" / "processed"

panel = pd.read_parquet(PROCESSED / "panel_trend.parquet")
print(f"panel_trend: {panel.shape[0]:,} rows x {panel.shape[1]} cols")

panel_clean: 1,258,888 rows x 66 cols


In [4]:
panel

,CERT,REPDTE,NAME,ASSET,DEP,RBC1AAJ,RBC1RWAJ,RBCRWAJ,RBCT1CER,RBCT1J,...,BOGZ1FL075035503Q,AGE_YEARS,pca_tier,is_distressed,is_healthy,quarters_to_onset,onset_4q,near_merge_exit,roa_artifact,fast_failure
0,8,1990-03-31,FLEET BANK OF MAINE,1856556,1530183.0,7.369650,9.39,10.758763,NaN,133230.0,...,108414.0,56.358658,well,False,True,NaN,0.0,False,False,False
1,8,1990-06-30,FLEET BANK OF MAINE,1881454,1529305.0,6.992111,9.33,10.712806,NaN,130354.0,...,107456.0,56.607803,well,False,True,NaN,0.0,False,False,False
2,8,1990-09-30,FLEET BANK OF MAINE,1914284,1502336.0,7.165424,9.53,10.906755,NaN,130911.0,...,107014.0,56.859685,well,False,True,NaN,0.0,False,False,False
3,8,1990-12-31,FLEET BANK OF MAINE,1809474,1470999.0,6.442283,9.06,10.452007,NaN,121369.0,...,106574.0,57.111567,well,False,True,NaN,0.0,False,False,False
4,8,1991-03-31,FLEET BANK OF MAINE,2738380,2330638.0,6.187796,8.20,9.532702,NaN,155581.0,...,105565.0,57.357974,adequate,False,True,NaN,0.0,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1258883,91393,1996-06-30,PROFESSIONAL BANK,88250,77173.0,11.108098,15.38,16.633992,NaN,9421.0,...,92642.0,13.325120,well,False,True,NaN,0.0,False,False,False
1258884,91393,1996-09-30,PROFESSIONAL BANK,88268,75840.0,11.293637,16.10,17.360053,NaN,9508.0,...,92134.0,13.577002,well,False,True,NaN,0.0,True,False,False
1258885,91393,1996-12-31,PROFESSIONAL BANK,93585,78195.0,10.530762,16.51,17.768309,NaN,9599.0,...,97377.0,13.828884,well,False,True,NaN,0.0,True,False,False
1258886,91393,1997-03-31,PROFESSIONAL BANK,85160,68073.0,11.319501,16.93,18.191136,NaN,9711.0,...,105096.0,14.075291,well,False,True,NaN,0.0,True,False,False


### Capital headroom — the untrained benchmark

The rule says a bank must hold at least 8% total risk-based capital (and meet three other
minimums). A bank at 12% has four points of room; a bank at 8.5% has half a point. Less
room means closer to crossing the line.

**Headroom = the smallest gap between any of a bank's capital ratios and its minimum.**
The smallest gap is used because that is how the regulation itself works — the worst ratio
sets the capital tier, so it is the binding constraint.

The applicable ratios change in 2015 (Basel III adds common equity tier 1 and raises the
Tier 1 minimum), so the thresholds switch with the date, matching the label built in
`feature_selection.ipynb`. Ratios that do not apply in a given era contribute nothing.

It is built here, before the split, so the benchmark travels with the test rows. It is a
benchmark and not a feature, so the guardrail below excludes it from the model inputs.

In [ ]:
# Minimums for "adequately capitalized" — below any of these is undercapitalized.
# Source: 12 CFR 325.103 (1990-2014) and 12 CFR 324.403 (2015+), per
# ../literature/pca_label_definition.md
THRESHOLDS = {
    "pre2015": {"RBCRWAJ": 8.0, "RBC1RWAJ": 4.0, "RBC1AAJ": 4.0},
    "basel3":  {"RBCRWAJ": 8.0, "RBC1RWAJ": 6.0, "RBC1AAJ": 4.0, "RBCT1CER": 4.5},
}
BASEL3_START = pd.Timestamp("2015-01-01")

pre_basel3 = panel["REPDTE"] < BASEL3_START
gaps = pd.DataFrame(index=panel.index)

for ratio in ["RBCRWAJ", "RBC1RWAJ", "RBC1AAJ", "RBCT1CER"]:
    # A ratio absent from a regime's dict gets a NaN threshold, so its gap is NaN and
    # min() skips it -- e.g. CET1 simply does not exist before 2015.
    limit = np.where(pre_basel3,
                     THRESHOLDS["pre2015"].get(ratio, np.nan),
                     THRESHOLDS["basel3"].get(ratio, np.nan))
    gaps[ratio] = panel[ratio] - limit

panel["headroom"] = gaps.min(axis=1)

print(f"headroom: {panel['headroom'].isna().mean() * 100:.2f}% missing")
print(panel["headroom"].describe(percentiles=[.05, .25, .5, .75]).round(2).to_string())

## Leakage guardrail

The target (`onset_4q`) asks whether a healthy bank falls to undercapitalized within the
**next four quarters**. That forward look is what makes this problem leak-prone: any column
or any split that lets a training row see past its own prediction quarter hands the model
the answer, and the score comes back beautiful and meaningless.

This section fixes the rules **once**, before a single model is fit:

1. which columns are allowed to be inputs,
2. how the data is cut by time,
3. assertions that fail loudly if either rule is broken.

Everything below this point can then use `FEATURES`, `train`, and `test` without
re-checking leakage each time.

### 1 · Columns that can never be inputs

Four groups have to come out of the feature list, for four different reasons:

| Group | Columns | Why it can't be an input |
|---|---|---|
| **Identifiers** | `CERT`, `REPDTE`, `NAME`, `ESTYMD` | Not predictors — a certificate number carries no risk information. Kept in the table for reference and for the time split |
| **Label parts** | `onset_4q`, `quarters_to_onset`, `pca_tier`, `is_distressed`, `is_healthy` | These *are* the answer, or what the answer was built from. `quarters_to_onset` literally records how soon distress arrived |
| **Retroactive macro** | `USREC` | The NBER recession flag is dated **after the fact** — the Dec 2007 recession start was announced Dec 2008. A `1` sitting in a 2008Q1 row is knowledge nobody had that quarter, arriving exactly when failures spike |
| **Future-built flags** | `near_merge_exit`, `fast_failure` | Both were built in `cleaning.ipynb` from outcomes only known afterward. `near_merge_exit` is positive **0.00%** of the time against a 0.74% baseline — a free "this bank is safe" signal |

Dropping `USREC` costs nothing: the other macro columns (rates, unemployment, credit
spreads, financial stress) carry the same business-cycle signal and *were* genuinely
published at the time.

**`roa_artifact` stays in.** It's computed from the current quarter's ROA only, so it looks
backward, not forward. It is the one flag that is safe as an input.

The two dropped flags are **not deleted** — they stay in `panel` so results can be reported
separately on the hard cases (`fast_failure`) and sensitivity checked both ways
(`near_merge_exit`), exactly as `cleaning.ipynb` intended.

In [5]:
# --- The drop list: one definition, used everywhere below -----------------------

# Identifiers and raw dates. Not predictors, but needed for the split and for reference.
ID_COLS = ["CERT", "REPDTE", "NAME", "ESTYMD"]

# The target and everything it was derived from.
LABEL_COLS = ["onset_4q", "quarters_to_onset", "pca_tier", "is_distressed", "is_healthy"]

# Flags built from information that only exists after the prediction quarter.
FUTURE_FLAGS = ["near_merge_exit", "fast_failure"]

# Macro series dated retroactively. NBER announces recession dates months to a year
# after they begin, so this column wasn't knowable in the quarter it marks.
LOOKAHEAD_MACRO = ["USREC"]

DROP_FROM_FEATURES = ID_COLS + LABEL_COLS + FUTURE_FLAGS + LOOKAHEAD_MACRO

TARGET = "onset_4q"
FEATURES = [c for c in panel.columns if c not in DROP_FROM_FEATURES]

# Text columns that will need encoding before any model that can't take strings.
CATEGORICAL = [c for c in FEATURES if str(panel[c].dtype) in ("object", "category")]

print(f"features kept:   {len(FEATURES)}")
print(f"columns dropped: {len(DROP_FROM_FEATURES)}  ->  {DROP_FROM_FEATURES}")
print(f"needs encoding:  {CATEGORICAL}")
print(f"roa_artifact kept as a feature: {'roa_artifact' in FEATURES}")

features kept:   55
columns dropped: 11  ->  ['CERT', 'REPDTE', 'NAME', 'ESTYMD', 'onset_4q', 'quarters_to_onset', 'pca_tier', 'is_distressed', 'is_healthy', 'near_merge_exit', 'fast_failure']
needs encoding:  []
roa_artifact kept as a feature: True


### 2 · Split by time, with a gap year

**Why not a random split.** Shuffling rows puts 2008 and 2023 on both sides of the line.
The model trains on the same crisis it is later tested on and scores near-perfectly on
nothing at all.

**Why a gap year.** A training row at 2015Q4 is labeled using outcomes through 2016Q4.
If the test period opened in 2016, that training label would already describe the test
period. The gap is set to a full **four quarters** — exactly the label's horizon — so
every training label resolves before the test window opens.

**Why the test set stops at 2025Q1.** The panel runs to 2026Q1, but a row needs four
following quarters to be labeled at all. From 2025Q2 on, `cleaning.ipynb` blanked the
unobservable negatives (concern #8) and kept only the positives that had already crossed —
leaving **12 rows that are 100% positive with no negatives at all**. Left in the test set
they quietly inflate every recall figure. 2025Q1 is the last fully observable quarter.

Those trailing quarters aren't wasted: they keep every predictor, so they become the
**"who looks risky right now"** scoring set for the final presentation — a demo output,
not a validated score.

| Window | Dates | Purpose |
|---|---|---|
| **Train** | 1990Q1 – 2015Q4 | Fit the model |
| **Gap** | 2016Q1 – 2016Q4 | Buffer. Never fit on, never scored |
| **Test** | 2017Q1 – 2025Q1 | Honest evaluation, includes the 2023 failures |
| **Score now** | 2025Q2 – 2026Q1 | Unlabelable. Demo predictions only |

In [6]:
# --- Time boundaries ------------------------------------------------------------

TRAIN_END  = "2015-12-31"   # last quarter used for fitting
GAP_END    = "2016-12-31"   # 4 quarters wide = the label's forward horizon
TEST_START = "2017-01-01"
TEST_END   = "2025-03-31"   # last quarter with a full 4-quarter outcome window

# Only rows that carry a real label can be trained or scored on. Blank onset_4q means
# either "already distressed" or "outcome window not observable" — neither is trainable.
labeled = panel[panel[TARGET].notna()]

train = labeled[labeled["REPDTE"] <= TRAIN_END]
test  = labeled[(labeled["REPDTE"] >= TEST_START) & (labeled["REPDTE"] <= TEST_END)]

# Held out on purpose. Exists to be excluded, not used.
gap = labeled[(labeled["REPDTE"] > TRAIN_END) & (labeled["REPDTE"] <= GAP_END)]

# No label yet, but every predictor is present -> the live "who looks risky now" set.
score_now = panel[(panel["REPDTE"] > TEST_END) & (panel[TARGET].isna())]

for name, df in [("train", train), ("gap", gap), ("test", test)]:
    print(f"{name:<6} {len(df):>10,} rows   "
          f"{df['REPDTE'].min().date()} to {df['REPDTE'].max().date()}   "
          f"{int(df[TARGET].sum()):>5,} positives ({df[TARGET].mean() * 100:.2f}%)")

print(f"{'score':<6} {len(score_now):>10,} rows   "
      f"{score_now['REPDTE'].min().date()} to {score_now['REPDTE'].max().date()}   "
      f"unlabeled (demo only)")

train   1,026,767 rows   1990-03-31 to 2015-12-31   8,404 positives (0.82%)
gap        24,180 rows   2016-03-31 to 2016-12-31      31 positives (0.13%)
test      166,329 rows   2017-03-31 to 2025-03-31     563 positives (0.34%)
score      17,697 rows   2025-06-30 to 2026-03-31   unlabeled (demo only)


### 3 · Assertions

The rules above are only worth something if breaking them is noisy. Each check below
corresponds to a specific way this project could leak:

1. **No leaky column survived** — catches a future-built flag sliding back into `FEATURES`
   after an edit upstream.
2. **Train ends before test begins** — catches an accidental shuffle or a bad date string.
3. **Training labels resolve before the test window** — the gap-year check, stated in terms
   of the label horizon rather than trusting the dates by eye.
4. **The rare-event rate survives in both halves** — catches rebalancing (`SMOTE`, class
   weights) applied before the split instead of to the training fold only.
5. **Every test quarter contains real negatives** — catches the positives-only tail from
   concern #8, and any future version of the same mistake.

If a cell below this one ever changes the splits, re-run this cell.

In [7]:
# --- Guardrail checks: each one fails loudly on a specific leak -------------------

# 1 · No dropped column made it into the feature list.
leaked = set(DROP_FROM_FEATURES) & set(FEATURES)
assert not leaked, f"leaky columns in FEATURES: {leaked}"

# 2 · Train and test never overlap in time.
assert train["REPDTE"].max() < test["REPDTE"].min(), "train and test windows overlap"

# 3 · Every training label resolves before the test window opens.
#     A row at quarter t is labeled from t+1..t+4, so push the last training date
#     forward by 4 quarters and check it still lands before the test starts.
last_train_label_resolves = train["REPDTE"].max() + pd.DateOffset(months=12)
assert last_train_label_resolves < test["REPDTE"].min(), (
    f"training labels resolve at {last_train_label_resolves.date()}, "
    f"inside the test window starting {test['REPDTE'].min().date()}"
)

# 4 · Both halves keep the true rare-event rate — no rebalancing has happened yet.
for name, df in [("train", train), ("test", test)]:
    rate = df[TARGET].mean()
    assert 0.001 < rate < 0.05, f"{name} positive rate {rate:.4f} is not the real base rate"

# 5 · Every test quarter has real negatives (guards the positives-only tail, concern #8).
quarters_without_negatives = [
    d.date() for d, s in test.groupby("REPDTE")[TARGET] if not (s == 0).any()
]
assert not quarters_without_negatives, (
    f"test quarters with no negatives: {quarters_without_negatives}"
)

print("all leakage checks passed")
print(f"  train: {train['REPDTE'].min().date()} to {train['REPDTE'].max().date()}")
print(f"  test:  {test['REPDTE'].min().date()} to {test['REPDTE'].max().date()}")
print(f"  gap:   {gap['REPDTE'].min().date()} to {gap['REPDTE'].max().date()} (excluded)")

all leakage checks passed
  train: 1990-03-31 to 2015-12-31
  test:  2017-03-31 to 2025-03-31
  gap:   2016-03-31 to 2016-12-31 (excluded)


### What this does not cover

Three leakage risks live in code that doesn't exist yet. They get handled where they arise,
not here:

- **Trend features.** The funding signals still to build (deposit growth, uninsured-deposit
  share, held-to-maturity losses) must use backward-only windows. A rolling window centred on
  the prediction quarter, or any `shift(-1)`, reads the future.
- **Feature ranking.** The single-feature scores (AUC) in `eda.ipynb` were computed across
  all years, test period included. Re-rank on `train` only, then freeze the list.
- **Filling blanks and scaling.** Fit on `train`, then apply to `test` — never a statistic
  computed over both. Tree models (gradient boosting) take blanks natively and sidestep this.

One remaining limitation, not a fix: the economic columns (`FRED`) are treated as available
in their own quarter, but several publish late — GDP (`GDPC1`) about a month after the
quarter closes, house prices (`USSTHPI`) and lending standards (`DRTSCILM`) later still, all
revised afterward. Milder than `USREC`, since the lag is short and fixed rather than a
committee decision. Lag the macro block one quarter to be strict, or note it in the writeup.

## The four things being compared

Every model below is judged against the same question: of the banks it ranks as riskiest,
how many actually became undercapitalized?

| Row | What it is | Trained? |
|---|---|---|
| **Naive floor** | Flag every bank | No |
| **Capital headroom** | Rank by how close the weakest capital ratio sits to its regulatory minimum | No |
| **Logistic regression** | Weights each feature, adds them into one score | Yes |
| **Gradient boosting** | Learns combinations of features | Yes |

The first two are reference points. The naive floor exists because Correia/Luck/Verner
(2024) report performance as a *multiple of* what flagging everything would achieve — a
model that cannot beat it has found nothing. The capital headroom row is the operational
status quo: Prompt Corrective Action *is* threshold monitoring, so this is roughly what
supervision already does without any model at all.

Gradient boosting is the **champion** — the method being defended. Logistic regression is
the **challenger**, the simpler and more explainable alternative it has to beat.

In [ ]:
# The benchmark score is derived from the capital ratios, which are already features.
# Leaving it in would hand the models the benchmark's own answer.
BENCHMARK_COLS = ["headroom"]

# Text columns are held back for this first run so both models see identical inputs.
# One-hot encoding state (56 levels) would give gradient boosting a different feature
# space than logistic regression, confounding the comparison. Noted as a follow-up.
FEATURES = [
    c for c in panel.columns
    if c not in DROP_FROM_FEATURES + BENCHMARK_COLS
    and str(panel[c].dtype) not in ("object", "category")
]

print(f"model features: {len(FEATURES)}")
print(f"  of which trend: {sum(1 for c in FEATURES if '_chg' in c or '_grow' in c)}")
print(f"held back (text, for later): {CATEGORICAL}")

## How performance is measured

Accuracy is useless here. Predicting "no distress" for every bank scores **99.66%** and
catches nothing. None of the ten papers reviewed uses it.

Four numbers instead, each answering a different question:

| Metric | Question it answers |
|---|---|
| **ROC-AUC** | Given one distressed and one healthy bank, how often is the distressed one ranked higher? 0.5 is a coin flip |
| **PR-AUC** | Of the banks flagged, what share are genuinely distressed? Sensitive to rarity, unlike ROC-AUC |
| **Lift** | PR-AUC divided by the base rate — how many times better than flagging everything. Correia's presentation |
| **Recall @ 1%** | With capacity to examine only the riskiest 1% of banks, what share of real cases are caught? |

Recall at a fixed alert budget is the one a risk committee actually asks about, and it
matches the asymmetric-cost framing in Cole & White: missing a failure costs far more than
a false alarm, but examiner time is finite.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

ALERT_BUDGET = 0.01  # examine the riskiest 1% of banks


def evaluate(name, scores, y, budget=ALERT_BUDGET):
    """Score one ranking. Higher `scores` must mean higher risk."""
    scores = np.asarray(scores, dtype=float)
    n_alerts = int(len(scores) * budget)
    flagged = np.argsort(-scores)[:n_alerts]

    pr_auc = average_precision_score(y, scores)
    return {
        "model": name,
        "ROC-AUC": roc_auc_score(y, scores),
        "PR-AUC": pr_auc,
        "lift": pr_auc / y.mean(),          # multiple of the naive floor
        "recall@1%": y[flagged].sum() / y.sum(),
    }

## Fitting the two models

**Logistic regression** cannot handle blanks or wildly different scales, so it gets a
pipeline: fill blanks with the median, then standardise. Both steps are fitted on the
**training data only** and then applied to test — fitting them on everything would leak
test-period information into training, the preprocessing leak flagged at the end of this
notebook.

**Gradient boosting** takes blanks natively and is scale-invariant, so it gets the raw
columns. That is one of the reasons it suits this data, where the era gaps are structural.

Neither model is tuned beyond sensible defaults. Tuning until the champion wins is how a
comparison becomes an argument for a foregone conclusion.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X_train, y_train = train[FEATURES], train[TARGET].values
X_test,  y_test  = test[FEATURES],  test[TARGET].values

# Columns with no observed value at all in the training window cannot be imputed;
# drop them for the linear model only. (CET1 is empty before 2015.)
LR_FEATURES = [c for c in FEATURES if X_train[c].notna().any()]

logistic = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000),
).fit(X_train[LR_FEATURES], y_train)

# class_weight="balanced" tells the model a missed case costs as much as the class is
# rare -- the rebalancing Petropoulos et al. (2020) achieve by downsampling instead.
# Applied to the training fit only; the test set keeps its true 0.34% rate.
boosting = HistGradientBoostingClassifier(
    max_iter=400,
    learning_rate=0.05,
    min_samples_leaf=50,
    l2_regularization=1.0,
    class_weight="balanced",
    random_state=0,
).fit(X_train, y_train)

print(f"logistic regression: {len(LR_FEATURES)} features")
print(f"gradient boosting:   {len(FEATURES)} features")

## Results — test period 2017Q1 to 2025Q1

In [ ]:
# Headroom is missing for a few rows; treat those as maximally safe so they rank last
# rather than being dropped from the comparison.
headroom_score = -test["headroom"].fillna(test["headroom"].max()).values

results = pd.DataFrame([
    evaluate("naive (flag everything)",
             np.random.RandomState(0).rand(len(y_test)), y_test),
    evaluate("capital headroom", headroom_score, y_test),
    evaluate("logistic regression",
             logistic.predict_proba(X_test[LR_FEATURES])[:, 1], y_test),
    evaluate("gradient boosting",
             boosting.predict_proba(X_test)[:, 1], y_test),
])

print(f"test rows {len(test):,} | positives {int(y_test.sum()):,} | base rate {y_test.mean():.4f}\n")
print(results.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

## Diagnostic — does it work when the failure mechanism matches?

The result above is weak, and the temptation is to tune until the champion wins. The more
useful question is *why* it is weak.

The training years (1990–2015) are dominated by credit-driven distress: bad loans, real
estate, slow deterioration. The 2017–2025 test period contains the 2021–23 wave, which was
driven by interest rates and deposit flight — a different mechanism. If that is the
explanation, the same model should perform well on a period whose failures *do* resemble
its training data.

So: train through 2005, skip 2006, and test on the 2008 crisis — which the model has never
seen, but which fails the same way its training years did.

In [ ]:
def run_split(train_end, test_start, test_end, label):
    """Refit everything on a different time split and return the comparison table."""
    tr = labeled[labeled["REPDTE"] <= train_end]
    te = labeled[(labeled["REPDTE"] >= test_start) & (labeled["REPDTE"] <= test_end)]
    y = te[TARGET].values

    lr_cols = [c for c in FEATURES if tr[c].notna().any()]
    lr = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                       LogisticRegression(max_iter=1000)).fit(tr[lr_cols], tr[TARGET])
    gb = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05,
                                        min_samples_leaf=50, l2_regularization=1.0,
                                        random_state=0).fit(tr[FEATURES], tr[TARGET])

    out = pd.DataFrame([
        evaluate("capital headroom", -te["headroom"].fillna(te["headroom"].max()).values, y),
        evaluate("logistic regression", lr.predict_proba(te[lr_cols])[:, 1], y),
        evaluate("gradient boosting", gb.predict_proba(te[FEATURES])[:, 1], y),
    ])
    print(f"\n{label}  |  train <= {train_end} ({len(tr):,})  ->  "
          f"test {test_start}..{test_end} ({len(te):,}, base {y.mean():.4f})")
    print(out.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    return out


crisis_2008 = run_split("2005-12-31", "2007-01-01", "2011-12-31", "2008 CREDIT CRISIS")

## Reading the two tables together

Same model, same features, same code — two different eras, opposite verdicts.

On the 2008 crisis the champion wins clearly and by a wide margin. On 2017–2025 it barely
edges the untrained benchmark, and loses on ROC-AUC outright.

That is not a broken model. It is a model that learned one failure mechanism well and was
then tested on another. Petropoulos et al. (2020) found the same shape of result: logistic
regression held up out-of-sample but was worst in six of eight criteria out-of-time, once
the period changed.

Two honest conclusions:

1. **The method is sound** — given failures that resemble what it trained on, it beats both
   the simple model and the status quo.
2. **The training data is the limitation, not the algorithm.** The trend features added in
   `feature_engineering.ipynb` were the first step toward the 2023 mechanism. They were not
   enough on their own, and the next step is the funding-side signals that Chu et al. (2026)
   and the SVB case study both point to.

## Diagnostics — locating where the model actually fails

The headline table is one number covering nine years, which hides more than it shows. Two
sweeps break it apart. Both are **diagnostics, not model selection**: the reported model
above is fixed, and nothing below is used to choose it. That distinction matters, because
running many variants and reporting the winner would quietly turn the test set into a
training set.

### Sweep 1 — does older training data help or hurt?

The training window runs from 1990, but the 1990s and the 2000s were different banking
industries under different capital rules. If the early years are teaching the wrong lesson,
cutting them should improve performance on a modern test period.

Nine training start years, one fixed test period.

In [ ]:
def train_from(start_year):
    """Refit the champion on a shortened training window. Test period never changes."""
    tr = train[train["REPDTE"] >= f"{start_year}-01-01"]
    m = HistGradientBoostingClassifier(
        max_iter=400, learning_rate=0.05, min_samples_leaf=50,
        l2_regularization=1.0, class_weight="balanced", random_state=0,
    ).fit(tr[FEATURES], tr[TARGET])
    row = evaluate(str(start_year), m.predict_proba(X_test)[:, 1], y_test)
    row["train_rows"], row["train_pos"] = len(tr), int(tr[TARGET].sum())
    return row


sweep_era = pd.DataFrame([train_from(y) for y in
                          [1990, 1993, 1996, 1999, 2002, 2005, 2008, 2010, 2012]])
sweep_era = sweep_era.rename(columns={"model": "train from"})
print(sweep_era[["train from", "train_rows", "train_pos",
                 "ROC-AUC", "PR-AUC", "recall@1%"]].to_string(
      index=False, float_format=lambda v: f"{v:.4f}"))

### Reading Sweep 1: the era does not matter much, and 2010 is a mirage

Everything from 1993 to 2005 lands in a narrow band around 0.070 PR-AUC. There is no cliff,
so there is no regime break hiding in the training data.

The one apparent standout, 2010, is worth looking at closely, because it is exactly the kind
of result that gets reported without scrutiny:

| Train from | PR-AUC |
|---|---|
| 2008 | 0.063 |
| **2010** | **0.078** |
| 2012 | 0.052 |

A genuine effect would trend. A single high point wedged between two low neighbours is
sampling noise — and by 2012 the training set holds only 370 positives, so the estimates
there are unstable to begin with. Cutting to 2010 would also discard the 2008 crisis, the
densest source of real distress cases in the data.

**Decision: keep the full 1990–2015 window.** Not because it scored best, but because the
sweep shows nothing is gained by cutting, and cutting invites a choice made on test results.

### Sweep 2 — when does the model stop working?

The same fixed model, scored one test year at a time. If performance degrades smoothly, the
problem is drift. If it falls off a cliff in specific years, something changed in those years.

Base rates differ year to year, so PR-AUC is also shown as **lift** — the multiple of what
flagging at random would achieve — which is comparable across rows.

In [ ]:
scored = test.copy()
scored["model_score"] = boosting.predict_proba(X_test)[:, 1]
scored["bench_score"] = -scored["headroom"].fillna(scored["headroom"].max())

rows = []
for year, block in scored.groupby(scored["REPDTE"].dt.year):
    y = block[TARGET].values
    if y.sum() < 3:                      # too few cases for a stable estimate
        continue
    model_pr = average_precision_score(y, block["model_score"])
    bench_pr = average_precision_score(y, block["bench_score"])
    rows.append({
        "year": year, "banks": len(block), "cases": int(y.sum()),
        "base rate": y.mean(),
        "ROC-AUC": roc_auc_score(y, block["model_score"]),
        "model lift": model_pr / y.mean(),
        "bench lift": bench_pr / y.mean(),
    })

sweep_year = pd.DataFrame(rows)
print(sweep_year.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

hard = sweep_year[sweep_year["year"].between(2021, 2023)]["cases"].sum()
print(f"\n2021-2023 holds {hard} of {int(y_test.sum())} test cases "
      f"({hard / y_test.sum() * 100:.0f}%)")

### Reading Sweep 2: the failure has a date

The model does not degrade. It works, stops working for three years, and starts working
again:

| Period | ROC-AUC | Verdict |
|---|---|---|
| 2017–2019 | 0.95–0.97 | Strong. Beats the benchmark roughly 2:1 in 2018 and 2019 |
| **2021–2023** | **0.53–0.75** | **Collapse.** 2021 is barely better than a coin flip |
| 2024–2025 | 0.81–0.92 | Recovered |

And the collapse lands exactly where the cases are: **2021–2023 holds 435 of the 563 test
cases, 77% of them.** That is why the headline number looked mediocre — the aggregate is
dominated by the one window the model cannot read.

This is the same shape of result Petropoulos et al. (2020) report: a model that validates
well within its own period and degrades once the period changes.

**What changed in those years.** The training data ends in 2015 and is dominated by
credit-driven distress — bad loans, real estate, gradual deterioration over two years, the
pattern in the EDA. The 2021–23 wave was driven by interest-rate shock and deposit flight:
fast, funding-side, and with capital ratios that *rose* into failure (the SVB case study in
`eda.ipynb`). The model was not taught that mechanism, and the funding features added in
`feature_engineering.ipynb` are a first step at it, not a solution.

**What this changes about the claim.** Not "gradient boosting beats the status quo" — it
does not, on average, over this test period. The defensible claim is narrower and more
useful: *the method works on the failure mechanism it was trained on, and the 2021–23 window
is a different mechanism, which is measurable, dateable, and the specific thing the next
round of features has to address.*

## Against the system actually in use

Every number so far compares the model to other models. A supervisor would ask a blunter
question: is this better than what the FDIC already runs?

There is a published answer. **SCOR** is the FDIC's off-site monitoring system, and it
predicts a *CAMELS downgrade* — the closest published analogue to this project's label, since
both ask whether a currently acceptable bank is about to be reclassified downward. Collier et
al. (2003), as quoted in Carmona et al. (2018), report its operating point:

> approximately two-thirds of institutions that were actually downgraded were **not**
> identified by the model, and approximately two-thirds of institutions that the model **did**
> identify for downgrades were not downgraded

In the vocabulary the FDIC papers use:

| | SCOR |
|---|---|
| **Type I — missed cases** | ~67% |
| **Type II — false alarms among those flagged** | ~67% |

That is the real bar, and it is a low one. It is also the honest way to state performance to
a risk committee: not an AUC, but *how many do we miss, and how much of our examiner time is
wasted*.

Below, the same two numbers are computed for each model across a range of alert budgets.

In [ ]:
def error_profile(name, scores, y, budgets=(0.005, 0.01, 0.02, 0.05, 0.10)):
    """Type I and Type II, in SCOR's terms, at several alert budgets."""
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores)
    out = []
    for b in budgets:
        k = int(len(scores) * b)
        flagged = order[:k]
        caught = y[flagged].sum()
        out.append({
            "model": name,
            "alert budget": f"{b:.1%}",
            "banks flagged": k,
            "Type I (missed)": 1 - caught / y.sum(),
            "Type II (false alarms)": 1 - caught / k,
        })
    return out


profiles = pd.DataFrame(
    error_profile("capital headroom", headroom_score, y_test)
    + error_profile("logistic regression",
                    logistic.predict_proba(X_test[LR_FEATURES])[:, 1], y_test)
    + error_profile("gradient boosting", boosting.predict_proba(X_test)[:, 1], y_test)
)

print("SCOR (Collier et al. 2003):  Type I ~67%   Type II ~67%\n")
print(profiles.to_string(index=False, float_format=lambda v: f"{v:.1%}"))

### What this comparison does and does not say

**The false-alarm rates are not comparable, and pretending otherwise would be dishonest.**
SCOR runs on banks already rated 1 or 2 and flags a few hundred; the base rate it works
against is far higher than the 0.34% here. Screening 166,000 banks for 563 cases produces
false-alarm rates that no supervisory system would tolerate, for arithmetic reasons alone —
at a 1% budget, catching every single case would still leave most alerts wrong.

**The miss rate is the number that transfers.** "What share of banks heading for trouble did
we fail to flag?" means the same thing in both systems, and SCOR's ~67% is a genuine
reference point for how hard this problem is even for the institution with the best data.

The comparison to make in the writeup is therefore one-sided and specific: **at what alert
budget does this model miss fewer cases than SCOR does**, and what does that budget cost in
examiner time. That is a claim a risk committee can act on, and it is defensible from the
published record rather than from an internal metric.

## Do the macro columns earn their place?

The panel joins eleven FRED series — rates, unemployment, GDP, house prices, credit spreads,
financial stress. That join was made on the assumption it was literature-backed. Re-reading
the source showed the opposite: Nuxoll (2003) ran exactly this test and concluded that
*"economic data do not improve these forecasts despite the fact that the data are
statistically significant."* Oshinsky & Olin (2005) excluded economic variables by choice.

So the join needs its own evidence. The test is an ablation — the same model, fitted twice,
with and without the macro block — which is the standard move in this literature (Nuxoll runs
it for economic data, Curry et al. for market data).

Run on both test periods, since a variable that is useless in calm years might still matter
in a crisis.

In [ ]:
MACRO = ["FEDFUNDS", "DGS10", "T10Y3M", "UNRATE", "GDPC1", "CPIAUCSL",
         "USSTHPI", "BAA10Y", "DRTSCILM", "NFCI", "BOGZ1FL075035503Q"]
WITHOUT_MACRO = [c for c in FEATURES if c not in MACRO]


def ablate(features, tr, te, label):
    m = HistGradientBoostingClassifier(
        max_iter=400, learning_rate=0.05, min_samples_leaf=50,
        l2_regularization=1.0, class_weight="balanced", random_state=0,
    ).fit(tr[features], tr[TARGET])
    return evaluate(label, m.predict_proba(te[features])[:, 1], te[TARGET].values)


train_2005 = labeled[labeled["REPDTE"] <= "2005-12-31"]
test_2008 = labeled[(labeled["REPDTE"] >= "2007-01-01") & (labeled["REPDTE"] <= "2011-12-31")]

ablation = pd.DataFrame([
    ablate(FEATURES,      train,      test,      "2017-2025  with macro"),
    ablate(WITHOUT_MACRO, train,      test,      "2017-2025  without macro"),
    ablate(FEATURES,      train_2005, test_2008, "2008 crisis  with macro"),
    ablate(WITHOUT_MACRO, train_2005, test_2008, "2008 crisis  without macro"),
])

print(f"{len(FEATURES)} features with macro, {len(WITHOUT_MACRO)} without\n")
print(ablation.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

### Verdict: Nuxoll was right, and still is

On the main test period the two models are **identical to four decimal places** — same PR-AUC,
same ROC-AUC to within 0.0003. Eleven features, no measurable contribution.

On the 2008 crisis the macro block adds about 4% of PR-AUC. Small, but the sign is consistent
with the idea that aggregate conditions matter when the whole system is moving together.

**Why this happens.** The macro columns hold the same eleven numbers for every bank in a given
quarter. They can raise or lower the predicted risk of the entire population at once, but they
carry no information about *which* bank is the problem — and picking which bank is the entire
task. That is Nuxoll's own explanation, and it holds on data running twenty-three years past
his.

**Decision: keep the block, and report this.** It costs nothing, it helps marginally in a
crisis, and the ablation is worth more in the writeup than the features are in the model — an
assumption inherited from a misread citation, tested directly, and resolved against a
published result.

The related caveat from the guardrail section still stands: several of these series publish
weeks after the quarter they describe, so a strict version would lag the whole block by one
quarter. Given they contribute nothing measurable, that refinement is not worth the complexity.